# 03 — Baseline: 5 models × 2 feature sets (RQ1, first look)
Default settings, 5% dataset (seed 42). **With checks** = all 63 features; **raw only** = the 13 data-quality check features removed. Logic: `src/models.py`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))     # project root, so `config` and `src` can be imported
import config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from src.features import feature_sets
from src.models import load_features, run_all, timing_test

X, y, issue = load_features(0.05, 42)
X_sets = feature_sets(X)
rate = y.mean()
print('Rows:', len(y), '| anomalies:', int(y.sum()), f'({rate:.2%})')
print({name: Xs.shape[1] for name, Xs in X_sets.items()}, 'features')

Every model uses the same stratified 70/30 split. Unsupervised models train on the training rows **without** labels and flag the `rate` share of rows; their anomaly score is used for PR-AUC. LOF and One-Class SVM use standardised features; supervised models use class weighting.

In [ ]:
results, per_issue = run_all(X_sets, y, issue, rate)
results.to_csv(config.RESULTS / 'baseline_results.csv', index=False)
per_issue.to_csv(config.RESULTS / 'baseline_recall_by_issue.csv')
results.round(3)

**PR-AUC** is the main metric (threshold-free; a random model scores about the anomaly rate, 0.05).

In [ ]:
pivot = results.pivot(index='model', columns='features', values='PR-AUC').loc[results['model'].unique()]
ax = pivot.plot.bar(figsize=(9, 4), rot=0)
ax.set_ylabel('PR-AUC'); ax.set_title('PR-AUC: with vs without data-quality check features')
ax.axhline(rate, ls='--', c='grey', lw=1, label='random baseline'); ax.legend()
plt.tight_layout(); plt.savefig(config.RESULTS / 'baseline_pr_auc.png', dpi=150); plt.show()

## Recall per issue type (`none` = false-alarm rate on clean rows)

In [ ]:
per_issue.round(2)

## Timing test: why not use all 1.09M rows?

In [ ]:
times, extrapolation = timing_test(StandardScaler().fit_transform(X_sets['raw only']))
print(times.round(2), '\n')
extrapolation.round(2)

In [ ]:
ax = times.plot(marker='o', figsize=(7, 4))
ax.set_xlabel('Rows'); ax.set_ylabel('Fit time (seconds)'); ax.set_title('Fit time vs sample size')
plt.tight_layout(); plt.savefig(config.RESULTS / 'timing_test.png', dpi=150); plt.show()

One-Class SVM grows roughly with the square of the rows (k ≈ 2); LOF grows almost as fast; Isolation Forest stays flat because each tree uses a small subsample. Times depend on the computer: report your own.